In [6]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense, Flatten, Dropout
from keras.applications import EfficientNetB0
import os, shutil

In [7]:
import kagglehub
path = kagglehub.dataset_download("irakozekelly/crop-pest-and-disease-dataset")
print("Path to dataset files:", path)

# Explorer immédiatement la structure
import os
print("\nContenu:", os.listdir(path))

Path to dataset files: /kaggle/input/datasets/irakozekelly/crop-pest-and-disease-dataset

Contenu: ['Dataset for Crop Pest and Disease Detection']


In [8]:
import os
from pathlib import Path

path = kagglehub.dataset_download("irakozekelly/crop-pest-and-disease-dataset")
base = f'{path}/Dataset for Crop Pest and Disease Detection/CCMT Dataset-Augmented'

merged = '/kaggle/working/merged'
crops = ['Cashew', 'Cassava', 'Maize', 'Tomato']

for split in ['train_set', 'test_set']:
    dest_split = 'train' if split == 'train_set' else 'val'
    for crop in crops:
        src = f'{base}/{crop}/{split}'
        for cls in os.listdir(src):
            dest = f'{merged}/{dest_split}/{crop}_{cls}'
            os.makedirs(dest, exist_ok=True)
            for img in os.listdir(f'{src}/{cls}'):
                os.symlink(f'{src}/{cls}/{img}', f'{dest}/{img}')

print("Fusion terminée")
print("Classes:", os.listdir(f'{merged}/train'))

Fusion terminée
Classes: ['Cassava_bacterial blight3241', 'Tomato_healthy', 'Cassava_healthy', 'Cashew_red rust4751', 'Maize_leaf spot', 'Tomato_leaf curl', 'Maize_fall armyworm', 'Cashew_leaf miner3466', 'Cashew_anthracnose3102', 'Maize_leaf beetle', 'Maize_leaf blight', 'Maize_streak virus', 'Tomato_leaf blight', 'Tomato_verticulium wilt', 'Cassava_green mite', 'Cashew_gumosis1714', 'Cassava_bacterial blight', 'Cashew_healthy5877', 'Cassava_brown spot', 'Maize_healthy', 'Cassava_mosaic', 'Tomato_septoria leaf spot', 'Maize_grasshoper']


In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    f'{merged}/train', labels='inferred', label_mode='int',
    batch_size=32, image_size=(224,224)
)
val_ds = keras.utils.image_dataset_from_directory(
    f'{merged}/val', labels='inferred', label_mode='int',
    batch_size=32, image_size=(224,224)
)

num_classes = len(train_ds.class_names)
print(f"{num_classes} classes: {train_ds.class_names}")

# Préparation optimale avec prefetch (la normalisation se fait désormais directement sur le GPU)
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

Found 80271 files belonging to 23 classes.


2026-07-02 18:05:00.556504: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Found 24981 files belonging to 22 classes.
23 classes: ['Cashew_anthracnose3102', 'Cashew_gumosis1714', 'Cashew_healthy5877', 'Cashew_leaf miner3466', 'Cashew_red rust4751', 'Cassava_bacterial blight', 'Cassava_bacterial blight3241', 'Cassava_brown spot', 'Cassava_green mite', 'Cassava_healthy', 'Cassava_mosaic', 'Maize_fall armyworm', 'Maize_grasshoper', 'Maize_healthy', 'Maize_leaf beetle', 'Maize_leaf blight', 'Maize_leaf spot', 'Maize_streak virus', 'Tomato_healthy', 'Tomato_leaf blight', 'Tomato_leaf curl', 'Tomato_septoria leaf spot', 'Tomato_verticulium wilt']


In [ ]:
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = True
set_trainable = False
for layer in base_model.layers:
    if layer.name == 'block6a_expand_conv':
        set_trainable = True
    layer.trainable = set_trainable

model = Sequential([
    # Cette ligne applique la division par 255.0 directement au niveau du GPU
    keras.layers.Rescaling(1./255, input_shape=(224, 224, 3)),
    base_model,
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=1e-5),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
history = model.fit(train_ds, epochs=20, validation_data=val_ds)

model.save('/kaggle/working/crop_disease_model.keras')

Epoch 1/8
 145/2509 ━━━━━━━━━━━━━━━━━━━━ 54:26 1s/step - accuracy: 0.0763 - loss: 3.4028